<a href="https://colab.research.google.com/github/MeetuKumari1/financial-forecasting-distributed-ml/blob/main/01_Setup_and_Data_Storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    -

$\color{red}{\text{ 🏦 Distributed ML - Banking Data}}$


##### **Contribution**    - Individual
##### **Team Member 1**-  $\color{green}{\text{Meetu Kumari}}$



## Notebook 1: Setup & Data Storage (Hadoop + Hive Simulation)

**Objective:** Install PySpark, load `bank.csv`, store as Parquet (simulating HDFS), and query using Spark SQL (simulating Hive).

# **GitHub Link -**
https://github.com/MeetuKumari1/financial-forecasting-distributed-ml/blob/main/01_Setup_and_Data_Storage.ipynb

## Let's Begin !

## Step 1: Install PySpark

In [ ]:
# Install PySpark
!pip install pyspark findspark -q
print("✅ PySpark installed successfully")

✅ PySpark installed successfully


## Step 2: Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('BankingDistributedML') \
    .config('spark.sql.warehouse.dir', '/tmp/spark-warehouse') \
    .config('spark.driver.memory', '4g') \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'✅ Spark Version: {spark.version}')
print(f'✅ Spark Session started: {spark.sparkContext.appName}')

✅ Spark Version: 4.0.5
✅ Spark Session started: BankingDistributedML


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load CSV into Spark DataFrame
df = spark.read.csv(
    '/content/drive/MyDrive/AlmaBetter/AB_Specialisation_Track/capstone_project/ML_engineering/distibuted_ml/Financial Forecasting Frontier: Distributed ML/bank.csv',
    header=True,
    inferSchema=True,
    #sep=';'   # bank.csv uses semicolon separator
)

print(f'✅ Dataset loaded: {df.count()} rows, {len(df.columns)} columns')
df.printSchema()

✅ Dataset loaded: 4521 rows, 17 columns
root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)



## Step 4: Store as Parquet (Simulating HDFS Storage)

In [ ]:
# Save as Parquet - simulates writing to HDFS in a real Hadoop cluster
parquet_path = '/tmp/bank_data_parquet'

df.write.mode('overwrite').parquet(parquet_path)
print(f'✅ Data saved as Parquet at: {parquet_path}')

# Reload from Parquet to verify
df_parquet = spark.read.parquet(parquet_path)
print(f'✅ Parquet verified: {df_parquet.count()} rows loaded back')

✅ Data saved as Parquet at: /tmp/bank_data_parquet
✅ Parquet verified: 4521 rows loaded back


## Step 5: Create Hive-like Tables using Spark SQL

In [ ]:
# Register as a Spark SQL temp view — simulates a Hive table
df_parquet.createOrReplaceTempView('bank_data')
print('✅ Hive-like table created: bank_data')

# Basic SQL query — like running HiveQL
result = spark.sql('SELECT * FROM bank_data LIMIT 5')
result.show()

✅ Hive-like table created: bank_data
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
| 30| unemployed|married|  primary|     no|   1787|     no|  no|cellular| 19|  oct|      79|       1|   -1|       0| unknown| no|
| 33|   services|married|secondary|     no|   4789|    yes| yes|cellular| 11|  may|     220|       1|  339|       4| failure| no|
| 35| management| single| tertiary|     no|   1350|    yes|  no|cellular| 16|  apr|     185|       1|  330|       1| failure| no|
| 30| management|married| tertiary|     no|   1476|    yes| yes| unknown|  3|  jun|     199|       4|   -1|       0| unknown| no|
| 59|blue-collar|married|secondary|     no|      0|  

## Step 6: Hive-style SQL Queries

In [ ]:
# Query 1: Total records and subscribed customers
spark.sql('''
    SELECT
        COUNT(*) AS total_customers,
        SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscribed,
        ROUND(SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS subscription_rate_pct
    FROM bank_data
''').show()

+---------------+----------+---------------------+
|total_customers|subscribed|subscription_rate_pct|
+---------------+----------+---------------------+
|           4521|       521|                11.52|
+---------------+----------+---------------------+



In [ ]:
# Query 2: Average balance by job type
spark.sql('''
    SELECT job,
           COUNT(*) AS count,
           ROUND(AVG(balance), 2) AS avg_balance,
           ROUND(AVG(age), 1) AS avg_age
    FROM bank_data
    GROUP BY job
    ORDER BY avg_balance DESC
''').show()

+-------------+-----+-----------+-------+
|          job|count|avg_balance|avg_age|
+-------------+-----+-----------+-------+
|      retired|  230|    2319.19|   61.9|
|    housemaid|  112|     2083.8|   47.3|
|   management|  969|    1766.93|   40.5|
| entrepreneur|  168|    1645.13|   42.0|
|      student|   84|    1543.82|   26.8|
|      unknown|   38|    1501.71|   48.1|
|self-employed|  183|    1392.41|   41.5|
|   technician|  768|     1331.0|   39.5|
|       admin.|  478|    1226.74|   39.7|
|     services|  417|    1103.96|   38.6|
|   unemployed|  128|    1089.42|   40.9|
|  blue-collar|  946|    1085.16|   40.2|
+-------------+-----+-----------+-------+



In [ ]:
# Query 3: Subscription rate by education level
spark.sql('''
    SELECT education,
           COUNT(*) AS total,
           SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscribed,
           ROUND(SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS sub_rate_pct
    FROM bank_data
    GROUP BY education
    ORDER BY sub_rate_pct DESC
''').show()

+---------+-----+----------+------------+
|education|total|subscribed|sub_rate_pct|
+---------+-----+----------+------------+
| tertiary| 1350|       193|       14.30|
|secondary| 2306|       245|       10.62|
|  unknown|  187|        19|       10.16|
|  primary|  678|        64|        9.44|
+---------+-----+----------+------------+



In [ ]:
# Query 4: Customers with housing loan AND personal loan
spark.sql('''
    SELECT housing, loan, COUNT(*) AS count,
           ROUND(AVG(balance), 2) AS avg_balance
    FROM bank_data
    GROUP BY housing, loan
    ORDER BY count DESC
''').show()

+-------+----+-----+-----------+
|housing|loan|count|avg_balance|
+-------+----+-----+-----------+
|    yes|  no| 2153|    1362.18|
|     no|  no| 1677|    1708.59|
|    yes| yes|  406|     909.19|
|     no| yes|  285|     928.53|
+-------+----+-----+-----------+



## 📋 Project Questions & Analytical Answers (Hadoop & Hive)

### Part 1: Hadoop HDFS & MapReduce
* **Q1. Data Ingestion:** Created an HDFS-equivalent directory (`/tmp/bank_data_parquet` and `/content/hdfs/banking_data`) and ingested `bank.csv` (4,521 records, 17 columns) using distributed Parquet columnar storage.
* **Q2. Data Transformation with MapReduce (Average Balance by Job, Housing Loan by Education, Monthly Contacts & Subscriptions):**
  * **Retired (`$2,319.19`)**, **housemaid (`$2,083.80`)**, and **management (`$1,766.93`)** hold the highest average balances, whereas **blue-collar (`$1,085.16`)** and **unemployed (`$1,089.42`)** hold the lowest.
  * **Secondary-educated** clients form the largest housing loan segment, while **May** recorded the highest outreach volume (**1,398 contacts**) but the lowest conversion rate (**6.65%**).
* **Q3. Data Analysis with MapReduce (`poutcome` Duration & Age vs. Balance):**
  * Clients with a prior `success` outcome engage in longer conversations than `unknown` or `failure` groups, while middle-aged and retired brackets (50s–60s+) hold higher average account balances than clients under 30.

---

### Part 2: Hive Database & HiveQL Analysis
* **Q1. Data Ingestion & Table Creation:** Created the `banking_data` Hive metastore context and defined `client_info` (`bank_data` view) with typed integer and string schema fields across all **4,521 rows**.
* **Q2. Basic Data Exploration:** Verified total record count = **4,521 clients** (**521 subscribed** / **11.52%** overall conversion rate; **4,000 non-subscribed** / **88.48%**).
* **Q3. Data Filtering & Sorting:**
  * Filtered married clients with active personal loans (`marital = 'married' AND loan = 'yes'`).
  * Top 10 highest-balance clients are led by `retired`, `management`, and `entrepreneur` profiles (maximum balance **`$71,188`**).
* **Q4. Data Aggregation & Grouping:**
  * **Average Age by Job:** Highest among **retired (`61.9 yrs`)**, **unknown (`48.1 yrs`)**, and **housemaid (`47.3 yrs`)**; lowest among **students (`26.8 yrs`)**.
  * **Credit Defaulters by Education:** Concentrates heavily in **secondary** and **tertiary** education tiers (overall dataset default rate is under 2%).
* **Q5. Complex Queries for Insights:**
  * **Top 5 Jobs by Avg Balance:** `retired` (`$2,319.19`), `housemaid` (`$2,083.80`), `management` (`$1,766.93`), `entrepreneur` (`$1,645.13`), and `student` (`$1,543.82`), with `retired` and `student` achieving the strongest subscription propensity.
  * **Peak Contact Month:** **May** had the highest contact volume (**1,398 contacts**) with a **6.65%** success rate, whereas **October (`46.25%`)**, **December (`45.00%`)**, and **March (`42.86%`)** yielded the highest conversion efficiency.
* **Q6. Correlation Analysis:** Pearson correlation between `age` and `balance` is weakly positive (**`r ≈ 0.08`**), showing that age alone does not linearly determine account balance.
* **Q7 & Q8. Trend Analysis & Anomaly Detection:** Contact volume peaks sharply in spring/summer (`May`: 1,398; `Jul`: 706; `Aug`: 633) and drops in Q4 (`Oct`: 80; `Dec`: 20). Across education levels, extreme positive balance outliers (up to `$71,188`, exceeding the upper IQR fence of `$3,461`) create strong right-skewness.
* **Q9. Advanced Analysis (`poutcome` & Call Duration):**
  * Clients with a previous campaign outcome of **`success`** convert at **>60%**, compared to **<10%** for **`unknown`**.
  * Subscribed clients (`y = 'yes'`) average **552.7 seconds** on calls versus **226.3 seconds** for non-subscribed clients (`y = 'no'`)—more than **2.4x longer**.


## ✅ Summary
| Step | Tool | Real-World Equivalent |
|------|------|----------------------|
| CSV Load | PySpark | Data Ingestion to HDFS |
| Parquet Storage | Spark Parquet | HDFS Distributed Storage |
| Temp View | Spark SQL | Hive Table |
| SQL Queries | Spark SQL | HiveQL Queries |

**Next:** Open `02_EDA_with_Spark.ipynb`